# 01. Exploratory Data Analysis

**Учебный проект.** Все клиенты и риски вымышлены. Главная задача: исследовать качество, структуру риска и изменения во времени, не смешивая будущие события с признаками.

Запускай Notebook из корня проекта.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

model = pd.read_csv(ROOT / "data/processed/model_dataset.csv", parse_dates=["SnapshotDate", "LabelReadyDate"])
if (ROOT / "data/raw/loans.csv").exists():
    loans = pd.read_csv(ROOT / "data/raw/loans.csv", parse_dates=["CreditOpenDate"])
    clients = pd.read_csv(ROOT / "data/raw/customers.csv")
    repayments = pd.read_csv(ROOT / "data/raw/repayment_schedule.csv", parse_dates=["DueDate", "SettledDate"])
else:
    print('Облегчённый набор: EDA по model_dataset. Для анализа событий платежей нужен полный архив.')
    loans = model[['CreditID', 'CustomerID', 'AgreementSumN']].copy()
    clients = model[['CustomerID']].drop_duplicates().copy()
    repayments = pd.DataFrame()
products = pd.read_csv(ROOT / "data/reference/product_catalog.csv")
print('Rows:', {"model": len(model), "loans": len(loans), "clients": len(clients), "payments": len(repayments)})
display(model.head())

## 1. Data Quality

Проверь уникальность ключей, типы, пропуски, датированность `SnapshotDate`, диапазоны сумм и связь таблиц. **Вопрос:** что означают `NaN` в Target и `LatePaymentRatio`?

In [ ]:
display(model.isna().mean().sort_values(ascending=False).head(15))
print('Duplicate CreditID:', model.CreditID.duplicated().sum())
print('Unlabeled loans:', model.TargetDPD30_12M.isna().sum())
print('Loan amount min/max:', loans.AgreementSumN.min(), loans.AgreementSumN.max())
# TODO: Проверь, что Rate и сроки соответствуют справочнику product_catalog.

## 2. Таргет и временные когорты

Проверяем не только общий Bad Rate, но и *vintage* по кварталам выдачи. Пропущенные Target не превращаем в Good.

In [ ]:
labeled = model[model.IsMatureTarget.eq(1)].copy()
labeled['OriginationQuarter'] = labeled.SnapshotDate.dt.to_period('Q').astype(str)
vintage = labeled.groupby('OriginationQuarter').agg(
    loans=('CreditID', 'size'), bad_rate=('TargetDPD30_12M', 'mean')
)
display(vintage.tail(12))
vintage.bad_rate.plot(figsize=(12, 4), marker='o', title='Доля DPD30+ по кварталам выдачи (синтетика)')
plt.ylabel('Доля кредитов с Target=1')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()
# TODO: Какие кварталы отличаются и почему? Учитывай число кредитов.

## 3. Продукты

**Гипотеза:** вероятность наблюдать DPD30+ в течение 12 месяцев может зависеть не только от риска клиента, но и от числа платежей и особенностей подпродукта.

In [ ]:
product_risk = labeled.groupby(['ProductFamily', 'ProductCode']).agg(
    n=('CreditID', 'size'), rate=('TargetDPD30_12M', 'mean'),
    avg_amount=('AgreementSumN', 'mean'), avg_term=('LoanTermMonths', 'mean')
).sort_values('rate', ascending=False)
display(product_risk)
# TODO: Построй график n и rate. Не делай выводы по группам с маленькой численностью.

## 4. География — только анализ гипотетического сценария

Повышенные значения для двух населённых пунктов введены в генератор. Они не отражают реальную платёжную дисциплину населения. Анализируй не только сырой Bad Rate, но и распределение продуктов, DTI, дохода. Город не является причинным объяснением просрочки.

In [ ]:
geo = labeled.groupby('City').agg(
    n=('CreditID', 'size'), bad_rate=('TargetDPD30_12M', 'mean'),
    mean_dti=('DebtToIncomeRatio', 'mean'), median_income=('IncomeMean6M_KGS', 'median')
).query('n >= 100').sort_values('bad_rate', ascending=False)
display(geo.head(12))
# TODO: Сравни две группы городов внутри Cash и Installment при одинаковых интервалах DTI.

## 5. Биннинг и разделяющая способность признаков

Отдели `NoCreditHistory=1` от `MaxHistoricalDPD=0`: это принципиально разные ситуации.

In [ ]:
labeled['dti_bin'] = pd.cut(labeled.DebtToIncomeRatio,
                            bins=[0, .2, .35, .5, .7, 1.0, np.inf], include_lowest=True)
dti_risk = labeled.groupby('dti_bin', observed=True).agg(
    n=('CreditID', 'size'), rate=('TargetDPD30_12M', 'mean')
)
display(dti_risk)
# TODO: Повтори анализ для IncomeCV6M, DPD30CountLast12M и CashflowMargin6M.

## 6. Data Leakage audit

Запрещено использовать `TargetDPD30_12M`, `LabelReadyDate`, `IsMatureTarget`, информацию об оплате после `SnapshotDate`, `CreditStatusAtCutoff` или итоги за будущий срок как X.

**Задание:** составь таблицу «признак / момент возникновения / доступен ли на T / риск утечки». Сохрани её рядом с EDA отчётом.

## Итоговые вопросы

1. Какие 5 признаков выглядят наиболее информативными для риска?
2. Что меняется после разделения по продуктам?
3. Почему нельзя автоматически считать всех клиентов без истории надёжными?
4. Какие гипотезы о региональном стресс-сценарии можно проверить, не приписывая городам причинный риск?
5. Какие улучшения нужны до обучения модели?